In [ ]:
# Session context: build role + project warehouse
from snowflake.snowpark.context import get_active_session
session = get_active_session()
for stmt in ["USE ROLE SC_ADMIN", "USE WAREHOUSE SC_WH"]:
    try:
        session.sql(stmt).collect()
    except Exception as e:
        print(f"{stmt} failed: {e}")
print(session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE()").collect())

In [ ]:
%%sql -r top_suppliers
WITH po AS (
    SELECT supplier_id, COUNT(DISTINCT po_id) AS n_po
    FROM SC.RAW_SUPPLIER.PURCHASE_ORDERS GROUP BY 1
), inco AS (
    SELECT supplier_id, MODE(incoterm) AS incoterm
    FROM SC.RAW_SUPPLIER.PURCHASE_ORDERS GROUP BY 1
), lt AS (
    SELECT supplier_id, ROUND(MEDIAN(lead_time_days)) AS lead_time_days
    FROM SC.RAW_SUPPLIER.SUPPLIER_PARTS GROUP BY 1
)
SELECT s.supplier_id, s.supplier_code, s.supplier_name, s.country_code, s.currency,
       s.payment_terms, i.incoterm, l.lead_time_days, p.n_po
FROM SC.RAW_SUPPLIER.SUPPLIERS s
JOIN po p USING (supplier_id)
LEFT JOIN inco i USING (supplier_id)
LEFT JOIN lt l USING (supplier_id)
ORDER BY p.n_po DESC, s.supplier_id
LIMIT 10

In [ ]:
import pandas as pd
df = top_suppliers.copy() if isinstance(top_suppliers, pd.DataFrame) else top_suppliers.to_pandas()
WEAK = {"S0011", "S0022"}  # Kronos Castings, Vela Polymers (see sql/01b_raw_supplier.sql)
assert WEAK <= set(df.SUPPLIER_ID), "weak suppliers missing from top 10"

# Synthetic contract terms (document content only, not a metric definition)
def terms(r):
    tier = int(r.SUPPLIER_ID[-1]) % 3
    return dict(
        otd_target=[95, 96, 98][tier],
        penalty_pct=[0.5, 1.0, 1.5][tier],
        penalty_cap=[5, 8, 10][tier],
        grace_days=[2, 1, 0][tier],
    )
contracts = [{**r._asdict(), **terms(r)} for r in df.itertuples(index=False)]
display(pd.DataFrame(contracts)[["SUPPLIER_ID","SUPPLIER_NAME","N_PO","otd_target","LEAD_TIME_DAYS","INCOTERM","PAYMENT_TERMS","penalty_pct"]])

In [ ]:
import importlib, subprocess, sys
backend = None
for mod, pkg in [("reportlab", "reportlab"), ("fpdf", "fpdf2")]:
    try:
        importlib.import_module(mod); backend = mod; break
    except ImportError:
        pass
if backend is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fpdf2"], capture_output=True, text=True, timeout=120)
    try:
        importlib.invalidate_caches(); importlib.import_module("fpdf"); backend = "fpdf"
    except ImportError:
        print("pip install failed -> using raw PDF writer")
        backend = "raw"
print("PDF backend:", backend)

In [ ]:
import os, datetime
OUT = "/tmp/contracts"
os.makedirs(OUT, exist_ok=True)
today = datetime.date.today().isoformat()

def lines_for(c):
    return [
        ("h", f"SUPPLY AGREEMENT - {c['SUPPLIER_NAME']}"),
        ("p", f"Contract No: CTR-{c['SUPPLIER_ID']}-2026    Effective: {today}    Term: 24 months"),
        ("p", f"Buyer: SC Manufacturing Group    Supplier: {c['SUPPLIER_NAME']} ({c['SUPPLIER_ID']}, {c['COUNTRY_CODE']})"),
        ("s", "1. Delivery Performance"),
        ("p", f"Contracted on-time delivery target: {c['otd_target']}% of PO lines, measured monthly."),
        ("p", f"Standard lead time: {int(c['LEAD_TIME_DAYS'])} calendar days from PO acknowledgement."),
        ("s", "2. Late-Delivery Penalty"),
        ("p", f"For each PO line delivered more than {c['grace_days']} day(s) after the confirmed date, Supplier"),
        ("p", f"shall credit Buyer {c['penalty_pct']}% of the line value per day late, capped at {c['penalty_cap']}% of"),
        ("p", "the line value. Credits are deducted from the next invoice. Force majeure excepted."),
        ("s", "3. Commercial Terms"),
        ("p", f"Incoterm: {c['INCOTERM']} (Incoterms 2020)    Currency: {c['CURRENCY']}"),
        ("p", f"Payment terms: {c['PAYMENT_TERMS']} from date of valid invoice."),
        ("s", "4. Signatures"),
        ("p", "Buyer: ______________________    Supplier: ______________________"),
    ]

def write_reportlab(path, items):
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas
    cv = canvas.Canvas(path, pagesize=A4); y = 800
    for kind, txt in items:
        cv.setFont("Helvetica-Bold" if kind != "p" else "Helvetica", {"h": 15, "s": 12, "p": 10}[kind])
        y -= 10 if kind == "s" else 0
        cv.drawString(50, y, txt); y -= 22 if kind == "h" else 16
    cv.save()

def write_fpdf(path, items):
    from fpdf import FPDF
    pdf = FPDF(); pdf.add_page()
    for kind, txt in items:
        pdf.set_font("Helvetica", "B" if kind != "p" else "", {"h": 15, "s": 12, "p": 10}[kind])
        if kind == "s": pdf.ln(3)
        pdf.cell(0, 8, txt, new_x="LMARGIN", new_y="NEXT")
    pdf.output(path)

def write_raw(path, items):
    esc = lambda s: s.replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")
    ops, y = [], 800
    for kind, txt in items:
        font, size = ("F2", {"h": 15, "s": 12}[kind]) if kind != "p" else ("F1", 10)
        y -= 10 if kind == "s" else 0
        ops.append(f"BT /{font} {size} Tf 50 {y} Td ({esc(txt)}) Tj ET"); y -= 22 if kind == "h" else 16
    stream = "\n".join(ops).encode("latin-1")
    objs = [b"<< /Type /Catalog /Pages 2 0 R >>",
            b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
            b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 595 842] /Resources << /Font << /F1 4 0 R /F2 5 0 R >> >> /Contents 6 0 R >>",
            b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
            b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica-Bold >>",
            b"<< /Length %d >>\nstream\n" % len(stream) + stream + b"\nendstream"]
    out, offs = bytearray(b"%PDF-1.4\n"), []
    for i, o in enumerate(objs, 1):
        offs.append(len(out)); out += b"%d 0 obj\n" % i + o + b"\nendobj\n"
    xref = len(out)
    out += b"xref\n0 %d\n0000000000 65535 f \n" % (len(objs) + 1)
    out += b"".join(b"%010d 00000 n \n" % o for o in offs)
    out += b"trailer\n<< /Size %d /Root 1 0 R >>\nstartxref\n%d\n%%%%EOF\n" % (len(objs) + 1, xref)
    open(path, "wb").write(out)

writer = {"reportlab": write_reportlab, "fpdf": write_fpdf, "raw": write_raw}[backend]
paths = []
for c in contracts:
    p = f"{OUT}/contract_{c['SUPPLIER_ID']}_{c['SUPPLIER_CODE']}.pdf"
    writer(p, lines_for(c)); paths.append(p)
for p in paths:
    print(p, os.path.getsize(p), "bytes")

In [ ]:
for p in paths:
    res = session.file.put(p, "@SC.RAW_DOCS.CONTRACTS", auto_compress=False, overwrite=True)
    print(res[0].source, res[0].status)

In [ ]:
%%sql -r stage_refresh
ALTER STAGE SC.RAW_DOCS.CONTRACTS REFRESH

In [ ]:
%%sql -r stage_listing
LIST @SC.RAW_DOCS.CONTRACTS